# Suite ADJ — Grid adjacency

An adjacency spec from one child to another moves the other beside it, along its rotation quantized to a quarter turn, where one change of the grid's sequence can, or into the cell beside it that a spec along the same side placed, which they then share; removing the spec reverts the child to its original insertion position: mbse-basis [CAP-239: grid adjacency](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-239), by the rules of [the diagrams proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/DIAGRAMS.md).

In [ ]:
import json

from mbse.Schemas.Framework import Builders, Validators
from mbse.Diagrams.Framework import Container, Logical

store = Builders.Store()
store.transaction(Logical.install)
store.transaction(Container.install)
L = store.Logical
diagrams = store.singleton("Logical.Diagrams")
problems = Validators.Validate(store).Reachable

with store.transaction():
    L.DiagramDirectory(diagrams, lambda f: f.diagrams(lambda a: a.insert(lambda e: e.name("overview").diagram(lambda d: d))))
overview = diagrams.diagrams({"name": "overview"})


def box(name, children, span, grows="rows", cell_rotation=0.0):
    """A top-level shape of the diagram holding a child of each name in `children`, laid out in a grid."""
    def spec(s):
        for child in children:
            s.children(lambda k, child=child: k.insert(lambda e: e.name(child).child(lambda c: c)))
        return s
    with store.transaction():
        L.OfDiagram(overview, lambda d: d.shapes(lambda s: s.insert(lambda e: e.name(name).shape(spec))))
        shape = overview.shapes({"name": name})
        Container.layout(shape, span, grows, cell_rotation)
    return shape


def beside(shape, a, b, degrees):
    """Adds the adjacency spec from child `a` to child `b`, by `degrees`, and adjoins them: whether it did."""
    with store.transaction():
        L.OfShape(shape.children({"name": a}), lambda s: s.adjacent_to(lambda x: x.insert(
            lambda e: e.to_shape(shape.children({"name": b})).rotation(lambda r: r.degrees(degrees)))))
        return Container.adjoined(shape.children({"name": a}), shape.children({"name": b}))


def apart(shape, a, b):
    """Removes the adjacency spec from child `a` to child `b`, and unjoins `b`."""
    with store.transaction():
        L.OfShape(shape.children({"name": a}), lambda s: s.adjacent_to(lambda x: x.remove(
            lambda e: e.to_shape(shape.children({"name": b})))))
        Container.unjoined(shape.children({"name": b}))


def cells(shape):
    """Each cell, in row and column order, as JSON text: its row and column, the names of the children it holds, in
    order, each with its order, and its axis and extents."""
    (entry,) = shape.grid
    names = {id(e.child): e.name.get() for e in shape.children}
    found = [[[e.row.get(), e.column.get()], [[names[id(h.shape)], h.order.get()] for h in e.cell.children],
              e.cell.grows.get(), e.cell.rows.get(), e.cell.columns.get()]
             for e in sorted(entry.grid.cells, key=lambda e: (e.row.get(), e.column.get()))]
    return json.dumps(found, separators=(",", ":"))


<a id="ADJ-01"></a>
## ADJ-01 · A rotation is quantized to the nearest quarter turn, half way to the clockwise one

In [ ]:
assert json.dumps([Container.quarter(d) for d in (0.0, 44.9, 45.0, 134.9, 190.0, 270.0, 315.0, -90.0, 359.5, 720.0)], separators=(',', ':')) == '["north","north","east","east","south","west","north","west","north","north"]'

<a id="ADJ-02"></a>
## ADJ-02 · A spec moves its child beside the other: east, then south, by one change of the sequence each

In [ ]:
cpu = box("cpu", "abcde", 3)
assert cells(cpu) == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["b",1]],null,1,1],[[0,2],[["c",2]],null,1,1],[[1,0],[["d",3]],null,1,1],[[1,1],[["e",4]],null,1,1]]'
assert json.dumps(beside(cpu, "a", "e", 90.0), separators=(',', ':')) == 'true'
assert cells(cpu) == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["e",4]],null,1,1],[[0,2],[["b",1]],null,1,1],[[1,0],[["c",2]],null,1,1],[[1,1],[["d",3]],null,1,1]]'
assert json.dumps(beside(cpu, "a", "d", 190.0), separators=(',', ':')) == 'true'
assert cells(cpu) == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["e",4]],null,1,1],[[0,2],[["b",1]],null,1,1],[[1,0],[["d",3]],null,1,1],[[1,1],[["c",2]],null,1,1]]'

<a id="ADJ-03"></a>
## ADJ-03 · Two specs from one child along one side share a cell, which grows along its grid's axis; a spec it cannot meet stays

In [ ]:
assert json.dumps(beside(cpu, "a", "c", 80.0), separators=(',', ':')) == 'true'
assert cells(cpu) == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["e",4],["c",2]],"rows",2,1],[[0,2],[["b",1]],null,1,1],[[1,0],[["d",3]],null,1,1]]'
io = box("io", "pqrst", 3)
assert json.dumps([beside(io, "p", "r", 180.0), beside(io, "p", "t", 180.0), beside(io, "s", "r", 90.0)], separators=(',', ':')) == '[true,true,false]'
assert cells(io) == '[[[0,0],[["p",0]],null,1,1],[[0,1],[["q",1]],null,1,1],[[0,2],[["s",3]],null,1,1],[[1,0],[["r",2],["t",4]],"rows",2,1]]'

<a id="ADJ-04"></a>
## ADJ-04 · Removing a spec reverts its child to its original insertion position, then applies the specs that remain

In [ ]:
apart(cpu, "a", "e")
assert cells(cpu) == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["c",2]],null,1,1],[[0,2],[["b",1]],null,1,1],[[1,0],[["d",3]],null,1,1],[[1,1],[["e",4]],null,1,1]]'
apart(cpu, "a", "c")
assert cells(cpu) == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["b",1]],null,1,1],[[0,2],[["c",2]],null,1,1],[[1,0],[["d",3]],null,1,1],[[1,1],[["e",4]],null,1,1]]'
apart(io, "p", "r")
assert cells(io) == '[[[0,0],[["p",0]],null,1,1],[[0,1],[["q",1]],null,1,1],[[0,2],[["r",2]],null,1,1],[[1,0],[["s",3]],null,1,1],[[1,1],[["t",4]],null,1,1]]'

<a id="ADJ-05"></a>
## ADJ-05 · A spec the grid cannot meet is ignored

In [ ]:
bus = box("bus", "abcde", 3)
assert json.dumps([beside(bus, "a", "e", 0.0), beside(bus, "c", "e", 90.0), beside(bus, "c", "e", 180.0)], separators=(',', ':')) == '[false,false,false]'
assert cells(bus) == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["b",1]],null,1,1],[[0,2],[["c",2]],null,1,1],[[1,0],[["d",3]],null,1,1],[[1,1],[["e",4]],null,1,1]]'

<a id="ADJ-06"></a>
## ADJ-06 · A grid that grows columns: east is a column on, north and south within a column

In [ ]:
col = box("col", "abcd", 2, "columns")
assert cells(col) == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["c",2]],null,1,1],[[1,0],[["b",1]],null,1,1],[[1,1],[["d",3]],null,1,1]]'
assert json.dumps(beside(col, "a", "d", 90.0), separators=(',', ':')) == 'true'
assert cells(col) == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["d",3]],null,1,1],[[1,0],[["b",1]],null,1,1],[[1,1],[["c",2]],null,1,1]]'
assert json.dumps(beside(col, "d", "b", 0.0), separators=(',', ':')) == 'false'
assert cells(col) == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["d",3]],null,1,1],[[1,0],[["b",1]],null,1,1],[[1,1],[["c",2]],null,1,1]]'

<a id="ADJ-07"></a>
## ADJ-07 · Original insertion positions outlive removals and additions; a grid placed on a container applies its specs

In [ ]:
alu = box("alu", "xyz", 3)
assert json.dumps(beside(alu, "y", "z", 270.0), separators=(',', ':')) == 'true'
assert cells(alu) == '[[[0,0],[["x",0]],null,1,1],[[0,1],[["z",2]],null,1,1],[[0,2],[["y",1]],null,1,1]]'
with store.transaction():
    L.OfShape(alu, lambda s: s.children(lambda k: k.remove(lambda e: e.name("x"))))
    Container.removed(alu)
assert cells(alu) == '[[[0,0],[["z",2]],null,1,1],[[0,1],[["y",1]],null,1,1]]'
with store.transaction():
    L.OfShape(alu, lambda s: s.children(lambda k: k.insert(lambda e: e.name("w").child(lambda c: c))))
    Container.added(alu.children({"name": "w"}))
assert cells(alu) == '[[[0,0],[["z",2]],null,1,1],[[0,1],[["y",1]],null,1,1],[[0,2],[["w",3]],null,1,1]]'
apart(alu, "y", "z")
assert cells(alu) == '[[[0,0],[["y",1]],null,1,1],[[0,1],[["z",2]],null,1,1],[[0,2],[["w",3]],null,1,1]]'
with store.transaction():
    L.OfDiagram(overview, lambda d: d.shapes(lambda s: s.insert(lambda e: e.name("late").shape(lambda x: x.children(
        lambda k: k.insert(lambda e: e.name("p").child(lambda c: c)).insert(lambda e: e.name("q").child(lambda c: c))
        .insert(lambda e: e.name("r").child(lambda c: c)))))))
    late = overview.shapes({"name": "late"})
    L.OfShape(late.children({"name": "p"}), lambda s: s.adjacent_to(lambda x: x.insert(
        lambda e: e.to_shape(late.children({"name": "r"})).rotation(lambda r: r.degrees(180.0)))))
    Container.layout(late, 2)
assert cells(late) == '[[[0,0],[["p",0]],null,1,1],[[0,1],[["q",1]],null,1,1],[[1,0],[["r",2]],null,1,1]]'
assert json.dumps([problems(L.OfShape, s) for s in (cpu, io, bus, col, alu, late)], separators=(',', ':')) == '[[],[],[],[],[],[]]'

<a id="ADJ-08"></a>
## ADJ-08 · A grid's cells are turned by its cell rotation, and with them the compasses of the shapes they hold

In [ ]:
rot = box("rot", "ab", 2, "rows", 90.0)
with store.transaction():
    for name in "ab":
        L.OfShape(rot.children({"name": name}), lambda s: s.origin(lambda a: a.insert(lambda e: e.point(lambda p: p)))
                  .borders(lambda a: a.insert(lambda e: e.name("n").point(lambda p: p)))
                  .children(lambda k: k.insert(lambda e: e.name("inner").child(lambda c: c.origin(
                      lambda a: a.insert(lambda e: e.point(lambda p: p)))))))
        child = rot.children({"name": name})
        L.OfPoint([e.point for e in child.origin][0], lambda p, child=child: p.outgoing(lambda a: a.insert(
            lambda e: e.end(child.borders({"name": "n"})).rotation(lambda r: r.degrees(90.0)))))
        Container.layout(child, 1, "columns", 45.0)
a = rot.children({"name": "a"})
inner = a.children({"name": "inner"})
assert json.dumps([Logical.compass(a), Logical.compass(a.borders({"name": "n"})), Logical.compass(inner)], separators=(',', ':')) == '[90.0,180.0,135.0]'
assert json.dumps(beside(rot, "a", "b", 180.0), separators=(',', ':')) == 'false'
assert cells(rot) == '[[[0,0],[["a",0]],null,1,1],[[0,1],[["b",1]],null,1,1]]'
with store.transaction():
    Container.rotate(rot, 270.0)
assert json.dumps([Logical.compass(a), Logical.compass(a.borders({"name": "n"})), Logical.compass(inner)], separators=(',', ':')) == '[270.0,0.0,315.0]'